# CSCE 40103 Module 4 - Assignment 3
## Malware Feature Clustering

**Dataset used:** PDFMalware2022.csv

Date - Jun 20 2026
Name - Bryant Baum


## 1. Setup



In [125]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import linkage, dendrogram

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

BENIGN_CSV = Path("staDynBenignLab.csv")
VX_HEAVEN_CSV = Path("staDynVxHeaven2698Lab.csv")
VIRUS_TOTAL_CSV = Path("staDynVt2955Lab.csv")

RANDOM_STATE = 40103

print("Setup complete. No package installation or internet access required.")

Setup complete. No package installation or internet access required.


## Load the Dataset

Load the PDFMalware2022.csv into data_frame


In [126]:
def load_dataset(local_csv) -> pd.DataFrame:
    # Load the PDF malware dataset without internet or package installation.
    if local_csv.exists():
        print(f"Loading local CSV file: {local_csv}")
        loaded_df = pd.read_csv(local_csv)
        loaded_df.columns = loaded_df.columns.str.strip()
    else:
        print("Local CSV not found")

    return loaded_df

# shape of df_benign

In [127]:
df_benign = load_dataset(BENIGN_CSV)
df_benign["source_label"] = "benign"
df_benign["binary_label"] = "benign"
print("df_benign Dataset loaded successfully.")
print("Shape:", df_benign.shape)
df_benign.head()

Loading local CSV file: staDynBenignLab.csv
df_benign Dataset loaded successfully.
Shape: (595, 1089)


/var/folders/pr/t00ymn8d41n4ym_4rfyv0yl80000gn/T/ipykernel_82316/2507160884.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_benign["source_label"] = "benign"
/var/folders/pr/t00ymn8d41n4ym_4rfyv0yl80000gn/T/ipykernel_82316/2507160884.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_benign["binary_label"] = "benign"


,,filename,Virtual,Offset,loc,Import,Imports,var,Forwarder,UINT,LONG,BOOL,WORD,BYTES,large,short,dd,db,dw,XREF,ptr,DATA,FUNCTION,extrn,byte,word,dword,char,DWORD,stdcall,arg,locret,asc,align,WinMain,unk,cookie,off,nullsub,DllEntryPoint,System32,dll,CHUNK,BASS,HMENU,DLL,LPWSTR,void,HRESULT,HDC,...,compile_date,pointer_to_symbol_table,number_of_symbols,size_of_optional_header,characteristics,magic,major_linker_version,minor_linker_version,size_init_data,size_uninit_data,section_alignment,file_alignment,major_operating_system_version,minor_operating_system_version,major_image_version,minor_image_version,major_subsystem_version,minor_subsystem_version,size_of_headers,subsystem,dll_characteristics,loader_flags,number_of_imports.1,AddressOfEntryPoint,SizeOfHeaders.1,CheckSum,size_of_stack_reserve,size_of_stack_commit,size_of_heap_reserve,size_of_heap_commit,image_base,Size_image,BaseOfCode,number_of_rva_and_sizes.1,number_of_IAT_entires.1,count_mutex,files_operations,count_file_read,count_file_written,count_file_exists,count_file_deleted,count_file_copied,count_file_renamed,count_regkey_written,count_regkey_deleted,count_file_opened,count_dll_loaded,label,source_label,binary_label
0,0,005aea0582da91b95d44f616c7e1257f6b97514445970...,0,0,1,0,0,0,0,0,0,0,0,0,0,0,421,52,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1343268103,0,0,224,258,267,10,10,10240,0,4096,512,6,2,6,2,6,0,1024,2,33088,0,5,12810,1024,55744,262144.0,8192.0,1048576.0,4096.0,4194304.0,36864,4096,16,85.0,0,0,0,0,0,0,0,0,0,0,0,0,0,benign,benign
1,1,006bcbf75d4dcf4da65f5b89e4ed1a77c2733f1a8d80b...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,120,26,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1343269433,0,0,224,258,267,10,10,8192,0,4096,512,6,2,6,2,6,2,1024,3,33088,0,4,10505,1024,62768,262144.0,8192.0,1048576.0,4096.0,4194304.0,28672,4096,16,94.0,0,0,0,0,0,0,0,0,0,0,0,0,0,benign,benign
2,2,022fdc3d34c83da8b8905fa04047127f51d0023cdea32...,0,0,1,0,0,0,0,0,0,0,0,0,0,0,153,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,998081825,0,0,224,271,267,7,0,8704,0,4096,512,5,1,5,1,4,0,1024,3,32768,0,5,5442,1024,22559,262144.0,4096.0,1048576.0,4096.0,16777216.0,20480,4096,16,43.0,0,0,0,0,0,0,0,0,0,0,0,0,0,benign,benign
3,3,03877b98c8be5f43a6ab6e0e7dad493082897aceb8025...,0,0,1,0,0,0,0,0,0,0,0,0,0,0,765,62,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1247529729,0,0,224,258,267,9,0,10752,0,4096,512,6,1,6,1,6,1,1024,3,33088,0,7,19525,1024,72145,262144.0,8192.0,1048576.0,4096.0,16777216.0,40960,4096,16,108.0,0,0,0,0,0,0,0,0,0,0,0,0,0,benign,benign
4,4,04bc9c75612c5489ad943b31e4bf55827b47c25cd2583...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,266,25,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1343268044,0,0,224,258,267,10,10,7168,0,4096,512,6,2,6,2,6,2,1024,2,33088,0,5,9788,1024,44894,262144.0,8192.0,1048576.0,4096.0,4194304.0,32768,4096,16,78.0,0,0,0,0,0,0,0,0,0,0,0,0,0,benign,benign


# shape of df_vx

In [128]:
df_vx = load_dataset(VX_HEAVEN_CSV)
df_vx["source_label"] = "malware_vxheaven"
df_vx["binary_label"] = "malware"
print("df_vx Dataset loaded successfully.")
print("Shape:", df_vx.shape)
df_vx.head()

Loading local CSV file: staDynVxHeaven2698Lab.csv
df_vx Dataset loaded successfully.
Shape: (2698, 1089)


/var/folders/pr/t00ymn8d41n4ym_4rfyv0yl80000gn/T/ipykernel_82316/3657177597.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_vx["source_label"] = "malware_vxheaven"
/var/folders/pr/t00ymn8d41n4ym_4rfyv0yl80000gn/T/ipykernel_82316/3657177597.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_vx["binary_label"] = "malware"


,Virtual,Offset,loc,Import,Imports,var,Forwarder,UINT,LONG,BOOL,WORD,BYTES,large,short,dd,db,dw,XREF,ptr,DATA,FUNCTION,extrn,byte,word,dword,char,DWORD,stdcall,arg,locret,asc,align,WinMain,unk,cookie,off,nullsub,DllEntryPoint,System32,dll,CHUNK,BASS,HMENU,DLL,LPWSTR,void,HRESULT,HDC,LRESULT,HANDLE,...,compile_date,pointer_to_symbol_table,number_of_symbols,size_of_optional_header,characteristics,magic,major_linker_version,minor_linker_version,size_init_data,size_uninit_data,section_alignment,file_alignment,major_operating_system_version,minor_operating_system_version,major_image_version,minor_image_version,major_subsystem_version,minor_subsystem_version,size_of_headers,subsystem,dll_characteristics,loader_flags,number_of_imports.1,AddressOfEntryPoint,SizeOfHeaders.1,CheckSum,size_of_stack_reserve,size_of_stack_commit,size_of_heap_reserve,size_of_heap_commit,image_base,Size_image,BaseOfCode,number_of_rva_and_sizes.1,number_of_IAT_entires.1,count_mutex,files_operations,count_file_read,count_file_written,count_file_exists,count_file_deleted,count_file_copied,count_file_renamed,count_regkey_written,count_regkey_deleted,count_file_opened,count_dll_loaded,label,source_label,binary_label
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,129,19,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,560097138,0,0,224,33167,267,2,25,2048,0,4096,512,1,0,0,0,3,10,1024,3,0,0,2,4096,1024,0,1048576.0,8192.0,1048576.0,4096.0,4194304.0,20480,4096,16,0.0,0,0,0,0,0,0,0,0,0,0,0,0,1,malware_vxheaven,malware
1,0,0,3,0,0,0,0,0,0,0,0,0,0,0,306,84,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,55964,0,0,224,270,267,3,10,29696,0,4096,4096,4,0,0,0,4,0,1024,2,0,0,6,51868,1024,99428,1048576.0,4096.0,1048576.0,4096.0,4194304.0,55964,4096,16,144.0,0,0,0,0,0,0,0,0,0,0,0,0,1,malware_vxheaven,malware
2,0,0,7,0,0,0,0,0,0,0,0,0,0,0,1114,387,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,2557891634,0,0,224,271,267,6,0,72192,0,4096,512,4,0,0,0,4,0,1024,2,0,0,1,94391,1024,0,1048576.0,4096.0,1048576.0,4096.0,4194304.0,94791,4096,16,33.0,0,0,0,0,0,0,0,0,0,0,0,0,1,malware_vxheaven,malware
3,0,0,172,0,0,0,0,0,0,0,0,0,0,0,6385,1122,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,21,0,0,0,0,0,0,0,0,0,0,...,1213499312,0,0,224,271,267,6,0,12288,0,4096,4096,4,0,1,0,4,0,4096,2,0,0,1,5480,4096,505359,1048576.0,4096.0,1048576.0,4096.0,4194304.0,475136,4096,16,77.0,0,5,1,0,4,0,0,0,34,0,1,5,1,malware_vxheaven,malware
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,2013931587,0,0,224,33166,267,2,25,6144,0,4096,512,1,0,0,0,3,10,1423,2,0,0,2,664,1423,0,1048576.0,8192.0,1048576.0,4096.0,4194304.0,24576,4096,16,0.0,0,0,0,0,0,0,0,0,0,0,0,0,1,malware_vxheaven,malware


# shape of df_vt

In [129]:
df_vt = load_dataset(VIRUS_TOTAL_CSV)
df_vt["source_label"] = "malware_virustotal"
df_vt["binary_label"] = "malware"
print("df_vt Dataset loaded successfully.")
print("Shape:", df_vt.shape)
df_vt.head()

Loading local CSV file: staDynVt2955Lab.csv


/var/folders/pr/t00ymn8d41n4ym_4rfyv0yl80000gn/T/ipykernel_82316/3112091182.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_vt["source_label"] = "malware_virustotal"
/var/folders/pr/t00ymn8d41n4ym_4rfyv0yl80000gn/T/ipykernel_82316/3112091182.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_vt["binary_label"] = "malware"


df_vt Dataset loaded successfully.
Shape: (2955, 1090)


,filename,Virtual,Offset,loc,Import,Imports,var,Forwarder,UINT,LONG,BOOL,WORD,BYTES,large,short,dd,db,dw,XREF,ptr,DATA,FUNCTION,extrn,byte,word,dword,char,DWORD,stdcall,arg,locret,asc,align,WinMain,unk,cookie,off,nullsub,DllEntryPoint,System32,dll,CHUNK,BASS,HMENU,DLL,LPWSTR,void,HRESULT,HDC,LRESULT,...,compile_date,pointer_to_symbol_table,number_of_symbols,size_of_optional_header,characteristics,magic,major_linker_version,minor_linker_version,size_init_data,size_uninit_data,section_alignment,file_alignment,major_operating_system_version,minor_operating_system_version,major_image_version,minor_image_version,major_subsystem_version,minor_subsystem_version,size_of_headers,subsystem,dll_characteristics,loader_flags,number_of_imports.1,AddressOfEntryPoint,SizeOfHeaders.1,CheckSum,size_of_stack_reserve,size_of_stack_commit,size_of_heap_reserve,size_of_heap_commit,image_base,Size_image,BaseOfCode,number_of_rva_and_sizes.1,number_of_IAT_entires.1,count_mutex,files_operations,count_file_read,count_file_written,count_file_exists,count_file_deleted,count_file_copied,count_file_renamed,count_regkey_written,count_regkey_deleted,count_file_opened,count_dll_loaded,label,source_label,binary_label
0,0000fa4d5b6a06b20aebb76a2d46fa8c122d8e70566eda...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,227,147,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1329663709,0,0,224,783,267,2,22,20480,109568,4096,512,4,0,6,0,4,0,1024,2,32768,0,8,17191,1024,434547,2097152.0,4096.0,1048576.0,4096.0,4.194304e+06,245760,4096,16,0.0,0,0,0,0,0,0,0,0,0,0,0,0,1,malware_virustotal,malware
1,00050248396dab9852e60e8afaa265483a99ead18e6edc...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,555,146,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1342061809,0,0,224,258,267,9,0,335360,0,4096,512,5,0,0,0,5,0,1024,2,33088,0,4,12863,1024,390624,1048576.0,4096.0,1048576.0,4096.0,4.194304e+06,389120,4096,16,92.0,0,0,0,0,0,0,0,0,0,0,0,0,1,malware_virustotal,malware
2,0005a6d64cba0a220f5621ad17afa52f7413cfe9b158bf...,0,0,4,0,0,0,0,0,0,0,0,0,0,0,3069,792,21,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1208111689,0,0,224,271,267,7,10,16384,0,4096,512,5,1,5,1,4,0,4096,2,32768,0,7,105404,4096,182821,262144.0,4096.0,1048576.0,4096.0,1.677722e+07,167936,4096,16,186.0,0,1,1,0,0,0,0,0,0,0,1,1,1,malware_virustotal,malware
3,0006253243ffb8901f1493540fd5b9fc8144a7d3d59fb8...,0,0,130,0,0,0,0,0,0,0,0,0,0,0,9406,977,2,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,14,0,0,0,0,0,0,0,0,0,...,1422330987,0,0,224,271,267,6,0,4096,106496,4096,512,4,0,0,0,4,0,4096,2,0,0,4,149056,4096,0,1048576.0,4096.0,1048576.0,4096.0,4.194304e+06,159744,110592,16,0.0,0,3,1,0,1,0,0,0,0,0,1,4,1,malware_virustotal,malware
4,000d9c048778d192525053f87d35ba40b0b5f0e5acb50d...,0,0,42,0,0,0,0,0,0,0,0,0,0,0,1923,627,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1495659583,0,0,240,34,523,9,0,57344,0,4096,512,5,2,0,0,5,2,1024,2,33088,0,3,51884,1024,231597,1048576.0,4096.0,1048576.0,4096.0,5.368709e+09,208896,4096,16,182.0,0,0,0,0,0,0,0,0,0,0,0,0,1,malware_virustotal,malware


## combine the three datasets

In [130]:
df_combined_full = pd.concat([df_benign, df_vx, df_vt], ignore_index=True)

print("Combined dataset created.")
print("Shape:", df_combined_full.shape)

print("\nRows per source_label:")
print(df_combined_full["source_label"].value_counts())

print("\nRows per binary_label:")
print(df_combined_full["binary_label"].value_counts())

df_combined_full.head()

Combined dataset created.
Shape: (6248, 1091)

Rows per source_label:
source_label
malware_virustotal    2955
malware_vxheaven      2698
benign                 595
Name: count, dtype: int64

Rows per binary_label:
binary_label
malware    5653
benign      595
Name: count, dtype: int64


,,filename,Virtual,Offset,loc,Import,Imports,var,Forwarder,UINT,LONG,BOOL,WORD,BYTES,large,short,dd,db,dw,XREF,ptr,DATA,FUNCTION,extrn,byte,word,dword,char,DWORD,stdcall,arg,locret,asc,align,WinMain,unk,cookie,off,nullsub,DllEntryPoint,System32,dll,CHUNK,BASS,HMENU,DLL,LPWSTR,void,HRESULT,HDC,...,number_of_symbols,size_of_optional_header,characteristics,magic,major_linker_version,minor_linker_version,size_init_data,size_uninit_data,section_alignment,file_alignment,major_operating_system_version,minor_operating_system_version,major_image_version,minor_image_version,major_subsystem_version,minor_subsystem_version,size_of_headers,subsystem,dll_characteristics,loader_flags,number_of_imports.1,AddressOfEntryPoint,SizeOfHeaders.1,CheckSum,size_of_stack_reserve,size_of_stack_commit,size_of_heap_reserve,size_of_heap_commit,image_base,Size_image,BaseOfCode,number_of_rva_and_sizes.1,number_of_IAT_entires.1,count_mutex,files_operations,count_file_read,count_file_written,count_file_exists,count_file_deleted,count_file_copied,count_file_renamed,count_regkey_written,count_regkey_deleted,count_file_opened,count_dll_loaded,label,source_label,binary_label,__vbaVarIndexLoad,SafeArrayPtrOfIndex
0,0.0,005aea0582da91b95d44f616c7e1257f6b97514445970...,0,0,1,0,0,0,0,0,0,0,0,0,0,0,421,52,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,224,258,267,10,10,10240,0,4096,512,6,2,6,2,6,0,1024,2,33088,0,5,12810,1024,55744,262144.0,8192.0,1048576.0,4096.0,4194304.0,36864,4096,16,85.0,0,0,0,0,0,0,0,0,0,0,0,0,0,benign,benign,NaN,NaN
1,1.0,006bcbf75d4dcf4da65f5b89e4ed1a77c2733f1a8d80b...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,120,26,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,224,258,267,10,10,8192,0,4096,512,6,2,6,2,6,2,1024,3,33088,0,4,10505,1024,62768,262144.0,8192.0,1048576.0,4096.0,4194304.0,28672,4096,16,94.0,0,0,0,0,0,0,0,0,0,0,0,0,0,benign,benign,NaN,NaN
2,2.0,022fdc3d34c83da8b8905fa04047127f51d0023cdea32...,0,0,1,0,0,0,0,0,0,0,0,0,0,0,153,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,224,271,267,7,0,8704,0,4096,512,5,1,5,1,4,0,1024,3,32768,0,5,5442,1024,22559,262144.0,4096.0,1048576.0,4096.0,16777216.0,20480,4096,16,43.0,0,0,0,0,0,0,0,0,0,0,0,0,0,benign,benign,NaN,NaN
3,3.0,03877b98c8be5f43a6ab6e0e7dad493082897aceb8025...,0,0,1,0,0,0,0,0,0,0,0,0,0,0,765,62,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,224,258,267,9,0,10752,0,4096,512,6,1,6,1,6,1,1024,3,33088,0,7,19525,1024,72145,262144.0,8192.0,1048576.0,4096.0,16777216.0,40960,4096,16,108.0,0,0,0,0,0,0,0,0,0,0,0,0,0,benign,benign,NaN,NaN
4,4.0,04bc9c75612c5489ad943b31e4bf55827b47c25cd2583...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,266,25,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,224,258,267,10,10,7168,0,4096,512,6,2,6,2,6,2,1024,2,33088,0,5,9788,1024,44894,262144.0,8192.0,1048576.0,4096.0,4194304.0,32768,4096,16,78.0,0,0,0,0,0,0,0,0,0,0,0,0,0,benign,benign,NaN,NaN


# Explanation

- Labels should not be used when clustering features because they are non numeric and because they will train the data on the answers, therefore skewing the results. 

## 2: Data Cleaning and Feature Selection

Prepare a numeric feature matrix for clustering. Remove label columns and any identifier-like
columns.

### Drop initial columns
Remove source_label, binary_label, file names, hashes, and identifiers from the feature matrix.

In [131]:
drop_cols = ["source_label", "binary_label", "filename", "label","", " ", "compile_date"]
# need to ignore errors because not all cols exist for all sources
df_combined = df_combined_full.drop(columns=drop_cols, errors="ignore")
print(df_combined.shape)

(6248, 1085)


### Drop columns with too many missing values

In [132]:
# Calculate the percentage of missing values per column
missing_pct = df_combined.isnull().mean()

# Keep columns with less than 30% missing data
columns_to_keep = missing_pct[missing_pct < 0.30].index
df_combined = df_combined[columns_to_keep]
print(df_combined.shape)

(6248, 1085)


### Drop low variance columns

In [133]:
# Drop columns where all values are identical
unique_counts = df_combined.nunique()
df_combined = df_combined.loc[:, unique_counts > 1]
print(df_combined.shape)

(6248, 243)


### Identify feature correlation

In [134]:
# Create a correlation matrix
corr_matrix = df_combined.corr().abs()

# Select the upper triangle of the correlation matrix
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find columns with a correlation greater than 0.85
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.85)]

df_combined = df_combined.drop(columns=to_drop)

### Look at the remaining columns and decide which ones to keep 

In [135]:
print(df_combined.shape)
print('\nColumns:')
print(df_combined.columns.tolist())
print('\nData types:')
display(df_combined.dtypes.to_frame('dtype'))
df_combined.head(20)

(6248, 102)

Columns:
['loc', 'UINT', 'BOOL', 'dd', 'db', 'dw', 'ptr', 'byte', 'word', 'arg', 'align', 'cookie', 'off', 'dll', 'int', '.rdata:', '.data:', '.text:', 'proc', 'fs', 'gs', 'bt', 'call', 'cmc', 'faddp', 'fchs', 'fild', 'fld', 'fstcw', 'fxch', 'jl', 'jmp', 'mov', 'mul', 'neg', 'rcl', 'rdtsc', 'rep', 'ret', 'rol', 'sar', 'setb', 'setle', 'shl', 'sidt', 'stc', 'xor', 'nop', 'SetCursorPos', 'ShellExecuteA', 'GetFileAttributesA', 'rand', 'ent_whole_file', 'ent_mean', 'ent_var', 'ent_max', 'ent_min', 'filesize', 'number_of_IAT_entires', 'number_of_rva_and_sizes', 'size_code', 'SizeOfHeaders', 'machine', 'number_of_sections.1', 'pointer_to_symbol_table', 'number_of_symbols', 'characteristics', 'major_linker_version', 'minor_linker_version', 'size_init_data', 'size_uninit_data', 'section_alignment', 'file_alignment', 'major_operating_system_version', 'minor_operating_system_version', 'major_image_version', 'minor_image_version', 'major_subsystem_version', 'minor_subsystem_version',

,dtype
loc,int64
UINT,int64
BOOL,int64
dd,int64
db,int64
...,...
count_file_copied,int64
count_regkey_written,int64
count_regkey_deleted,int64
count_file_opened,int64


,loc,UINT,BOOL,dd,db,dw,ptr,byte,word,arg,align,cookie,off,dll,int,.rdata:,.data:,.text:,proc,fs,gs,bt,call,cmc,faddp,fchs,fild,fld,fstcw,fxch,jl,jmp,mov,mul,neg,rcl,rdtsc,rep,ret,rol,sar,setb,setle,shl,sidt,stc,xor,nop,SetCursorPos,ShellExecuteA,...,ent_whole_file,ent_mean,ent_var,ent_max,ent_min,filesize,number_of_IAT_entires,number_of_rva_and_sizes,size_code,SizeOfHeaders,machine,number_of_sections.1,pointer_to_symbol_table,number_of_symbols,characteristics,major_linker_version,minor_linker_version,size_init_data,size_uninit_data,section_alignment,file_alignment,major_operating_system_version,minor_operating_system_version,major_image_version,minor_image_version,major_subsystem_version,minor_subsystem_version,subsystem,dll_characteristics,loader_flags,number_of_imports.1,AddressOfEntryPoint,CheckSum,size_of_stack_reserve,size_of_stack_commit,size_of_heap_reserve,size_of_heap_commit,Size_image,BaseOfCode,count_mutex,files_operations,count_file_read,count_file_written,count_file_exists,count_file_deleted,count_file_copied,count_regkey_written,count_regkey_deleted,count_file_opened,count_dll_loaded
0,1,0,0,421,52,0,0,1,0,0,0,0,0,0,436,0,0,1,0,0,0,0,251,1,0,0,0,0,0,0,0,125,648,1,3,0,0,0,72,0,3,0,0,2,0,1,98,0,0,0,...,3.438467,2.536075,0.022814,2.739660,2.208557,57434,85.0,16,10752,1024,332,5,0,0,258,10,10,10240,0,4096,512,6,2,6,2,6,0,2,33088,0,5,12810,55744,262144.0,8192.0,1048576.0,4096.0,36864,4096,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,120,26,0,0,0,0,0,0,0,0,0,170,0,0,1,0,0,3,0,351,0,0,0,0,0,0,0,0,44,308,0,2,0,0,0,34,0,0,0,0,0,0,0,47,0,0,0,...,3.527134,2.584668,0.008117,2.733124,2.346066,44420,94.0,16,8192,1024,332,5,0,0,258,10,10,8192,0,4096,512,6,2,6,2,6,2,3,33088,0,4,10505,62768,262144.0,8192.0,1048576.0,4096.0,28672,4096,0,0,0,0,0,0,0,0,0,0,0
2,1,0,0,153,5,0,0,0,0,0,0,0,0,0,6,0,0,1,0,12,21,0,54,0,0,0,0,0,0,0,2,39,130,0,0,0,0,1,14,1,0,0,0,1,0,0,21,0,0,0,...,3.149796,2.458683,0.029404,2.710182,2.221808,20706,43.0,16,3072,1024,332,3,0,0,271,7,0,8704,0,4096,512,5,1,5,1,4,0,3,32768,0,5,5442,22559,262144.0,4096.0,1048576.0,4096.0,20480,4096,0,0,0,0,0,0,0,0,0,0,0
3,1,0,0,765,62,0,0,0,0,0,0,0,0,0,446,0,0,1,0,24,72,0,365,18,0,0,0,0,0,0,10,158,879,0,2,0,0,1,77,0,25,0,0,6,0,0,164,2,0,0,...,3.613366,2.616575,0.008031,2.701594,2.378513,69110,108.0,16,19456,1024,332,4,0,0,258,9,0,10752,0,4096,512,6,1,6,1,6,1,3,33088,0,7,19525,72145,262144.0,8192.0,1048576.0,4096.0,40960,4096,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,266,25,0,0,0,0,0,0,0,0,0,407,0,0,1,0,6,6,0,163,0,0,0,0,0,0,0,0,65,504,1,1,0,0,0,55,0,1,0,0,0,0,0,74,0,0,0,...,3.496844,2.562597,0.017237,2.752913,2.283647,39574,78.0,16,8192,1024,332,6,0,0,258,10,10,7168,0,4096,512,6,2,6,2,6,2,2,33088,0,5,9788,44894,262144.0,8192.0,1048576.0,4096.0,32768,4096,0,0,0,0,0,0,0,0,0,0,0
5,7,0,0,2564,103,10,0,0,0,0,0,0,0,0,622,0,0,1,0,38,58,0,475,0,0,0,1,0,0,0,43,209,1069,0,2,0,0,17,120,0,2,0,0,8,0,0,93,4,0,0,...,3.396606,2.532329,0.033073,2.751664,2.228936,97890,79.0,16,28672,1024,332,3,0,0,271,7,10,5120,0,4096,512,5,2,5,2,5,0,3,32768,0,6,24583,63187,262144.0,8192.0,1048576.0,4096.0,40960,4096,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,1415,85,0,0,1,0,0,0,0,0,0,456,0,0,1,0,0,8,0,315,0,0,0,0,0,0,0,0,111,861,0,4,0,0,0,90,0,21,0,0,0,0,0,167,0,0,0,...,3.454981,2.525830,0.027740,2.724341,2.177518,71640,78.0,16,17920,1024,332,5,0,0,258,10,10,8192,0,4096,512,6,2,6,2,6,2,3,33088,0,5,20233,33304,262144.0,8192.0,1048576.0,4096.0,40960,4096,0,0,0,0,0,0,0,0,0,0,0
7,1,0,0,808,104,4,0,0,0,0,0,0,0,0,534,0,0,1,0,37,67,0,443,0,0,0,1,0,0,0,35,140,936,3,41,0,0,25,109,0,0,0,0,2,0,1,109,0,0,0,...,3.501817,2.580283,0.032116,2.739550,1.974258,69440,84.0,16,20480,4096,332,3,0,0,271,7,10,12288,0,4096,4096,5,1,5,1,4,0,2,32768,0,7,10098,75614,262144.0,4096.0,1048576.0,4096.0,36864,4096,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,213,41,0,0,0,0,0,0,0,0,0,366,0,0,1,0,0,0,0,230,0,0,0,0,0,0,0,1,67,459,1,1,0,0,0,51,0,0,0,0,0,0,0,66,0,0,0,...,3.647083,2.615283,0.037527,2.844363,2.138362,181534,114.0,16,7680,1024,332,5,0,0,258,10,10,64000,0,4096,512,6,2,6,2,6,2,2,33088,0,10,10

In [140]:
features_keep = [
    "filesize", 
    "number_of_IAT_entires",
    "size_code",
    "size_init_data",
    "Size_image",
    "BaseOfCode",
    "files_operations",
    "dd",
    "db",
    "ret",
    "rol",
    "sar",
    "shl",
    "xor",
    "nop",
    "size_of_stack_reserve",
    "size_of_stack_commit"
]

df_features = df_combined[features_keep].copy()
print(df_features.shape)
df_features.head()


(6248, 17)


,filesize,number_of_IAT_entires,size_code,size_init_data,Size_image,BaseOfCode,files_operations,dd,db,ret,rol,sar,shl,xor,nop,size_of_stack_reserve,size_of_stack_commit
0,57434,85.0,10752,10240,36864,4096,0,421,52,72,0,3,2,98,0,262144.0,8192.0
1,44420,94.0,8192,8192,28672,4096,0,120,26,34,0,0,0,47,0,262144.0,8192.0
2,20706,43.0,3072,8704,20480,4096,0,153,5,14,1,0,1,21,0,262144.0,4096.0
3,69110,108.0,19456,10752,40960,4096,0,765,62,77,0,25,6,164,2,262144.0,8192.0
4,39574,78.0,8192,7168,32768,4096,0,266,25,55,0,1,0,74,0,262144.0,8192.0


### Check Missing Values and Duplicate Rows

Check whether the dataset contains missing values and duplicate rows and then remove duplicates

In [141]:
missing_summary = df_features.isna().sum().sort_values(ascending=False)
duplicate_count = df_features.duplicated().sum()

print('Duplicate rows:', duplicate_count)
print('\nMissing values:')
display(missing_summary[missing_summary > 0].to_frame('missing_count'))

if missing_summary.sum() == 0:
    print('No missing values found.')

Duplicate rows: 224

Missing values:


,missing_count


No missing values found.


### Feature summary

In [142]:
def feature_summary(feature, reason, meaning):
    return {
        "Feature": feature,
        "Reason for Keeping It": reason,
        "Possible Malware-Analysis Meaning": meaning
    }

summary_rows = []
for feature in features_keep:
    if feature == "dd" or feature == "db":
        summary_rows.append(feature_summary(feature, "Data directives that can give insight into file", "Can indicate packing, encryption, obfuscation"))
    elif feature == "xor" or feature == "ret" or feature == "nop":
        summary_rows.append(feature_summary(feature, "Assembly instructions", "High/low frequencies can indicate malware"))
    elif feature == "filesize" or feature == "size_code" or feature == "files_operations":
        summary_rows.append(feature_summary(feature, "Insight into file structure and behavior", "Abnormal structure behavior indicative of malware"))
    else:
        summary_rows.append(feature_summary(feature, "Randomly chose on visual inspection from remaining columns", "File characteristic"))

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,Feature,Reason for Keeping It,Possible Malware-Analysis Meaning
0,filesize,Insight into file structure and behavior,Abnormal structure behavior indicative of malware
1,number_of_IAT_entires,Randomly chose on visual inspection from remai...,File characteristic
2,size_code,Insight into file structure and behavior,Abnormal structure behavior indicative of malware
3,size_init_data,Randomly chose on visual inspection from remai...,File characteristic
4,Size_image,Randomly chose on visual inspection from remai...,File characteristic
5,BaseOfCode,Randomly chose on visual inspection from remai...,File characteristic
6,files_operations,Insight into file structure and behavior,Abnormal structure behavior indicative of malware
7,dd,Data directives that can give insight into file,"Can indicate packing, encryption, obfuscation"
8,db,Data directives that can give insight into file,"Can indicate packing, encryption, obfuscation"
9,ret,Assembly instructions,High/low frequencies can indicate malware


## Inspect the Target Variable

Display class counts and percentages, then create a class distribution plo

In [ ]:
class_counts = df["Class"].value_counts()
class_percentages = df["Class"].value_counts(normalize=True) * 100
print("Class counts:")
print(class_counts)
print("Class percentages:")
print(class_percentages.round(2))
sns.countplot(data=df, x="Class")
plt.title("Class Distribution")
plt.xlabel("Class: 0 = Benign, 1 = Malicious")
plt.ylabel("Number of Records")
plt.show()

## Explanation

Each row in the dataset describes a file. Each row has various file characteristics, like 'Pages', 'IsEncrypted', 'Text', 'Header', but also includes the Class labels 'Malicious' or 'Benign' as well as an index label. The Class label should be removed since it will create an obvious distinction when clustering. I will probably remove the index label as well since it does not really seem to provide any value towards the actual classification of the file being malicious or benign. 

## 3. Clean and Prepare Features


In [ ]:
cleaned_df = df.copy()

cleaning_notes = []
drop_cols = ["FileName", "Class", "__index_level_0__"]
feature_df = cleaned_df.drop(columns=drop_cols)
print(f"Dropped identifier/label columns: {drop_cols}")

if "Text" in cleaned_df:
    feature_df["Text"] = feature_df["Text"].map({"Yes": 1, "No": 0})

if "Header" in feature_df:
    feature_df["Header"] = feature_df["Header"].str.extract(r"(\d\.\d)").astype(float)

feature_df = feature_df.apply(pd.to_numeric, errors="coerce")
feature_df = feature_df.fillna(feature_df.median(numeric_only=True))

print('Dataset source: PDFMalware2022.csv')
print('Shape:', feature_df.shape)
print('\nColumns:')
print(feature_df.columns.tolist())
print('\nData types:')
display(feature_df.dtypes.to_frame('dtype'))
feature_df.head()


# 4. Scale Features and Apply PCA

In [ ]:
df_scaled = StandardScaler().fit_transform(feature_df)
pca = PCA(n_components=2, random_state=40103)
X_pca = pca.fit_transform(df_scaled)

pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])

label_col = "Class"
if label_col:
    pca_df[label_col] = df[label_col].astype(str).values

print("Explained variance ratio:", pca.explained_variance_ratio_)

plt.figure(figsize=(7, 5))
if label_col:
    sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue=label_col, s=55)
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
else:
    sns.scatterplot(data=pca_df, x="PC1", y="PC2", s=55)
plt.title("PCA View of Malware Feature Vectors")
plt.tight_layout()
plt.show()

# Explanation

- It is important to scale features when training with K-means, hierarchical clustering, DBSCAN, or silhouette score, because these algorithms require calculating features by 'distance' and scaling makes sure that each feature is contributing a balanced proportion of magnitude rather than features with large values dominating the space. 

## 5. K-Means Clustering

In [ ]:
def evaluate_k_range(X_scaled, k_values, random_state=40103):
    rows = []
    for k in k_values:
        model = KMeans(
            n_clusters=k,
            max_iter=300,
            n_init=20,
            random_state=random_state
        )
        labels = model.fit_predict(X_scaled)
        inertia = model.inertia_
        sil = np.nan
        if k >= 2:
            sil = silhouette_score(X_scaled, labels)
        rows.append({'K': k, 'inertia': inertia, 'silhouette': sil})
    return pd.DataFrame(rows)

k_values = range(2, 9)
k_eval = evaluate_k_range(df_scaled, k_values, RANDOM_STATE)
display(k_eval.round(3))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_eval['K'], k_eval['inertia'], marker='o')
plt.xlabel('Number of clusters, K')
plt.ylabel('Inertia')
plt.title('Elbow Method for K-Means')
plt.xticks(k_eval['K'])
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def plot_clusters_pca(X_scaled, cluster_labels, title):
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_scaled)

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, alpha=0.75)
    plt.xlabel("PCA component 1")
    plt.ylabel("PCA component 2")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.colorbar(scatter, label="K-Means cluster")
    plt.show()

    print("Explained variance ratio:", np.round(pca.explained_variance_ratio_, 3))

# auto picks the k with the best silhouette value
CHOSEN_K = int(k_eval.sort_values("silhouette", ascending=False).iloc[0]["K"])
print("Chosen K:", CHOSEN_K)

final_kmeans = KMeans(
    n_clusters=CHOSEN_K,
    max_iter=300,
    n_init=20,
    random_state=RANDOM_STATE
)
cluster_labels = final_kmeans.fit_predict(df_scaled)
df["kmeans_cluster"] = cluster_labels

print("Cluster counts:")
print(pd.Series(cluster_labels).value_counts().sort_index())
print("\nInertia:", round(final_kmeans.inertia_, 3))
print("Silhouette score:", round(silhouette_score(df_scaled, cluster_labels), 3))

if label_col:
    print("\nKnown Class vs K-Means cluster:")
    display(pd.crosstab(df[label_col], df["kmeans_cluster"], rownames=["Class"], colnames=["Cluster"]))

plot_clusters_pca(df_scaled, cluster_labels, f"K-Means clusters (K={CHOSEN_K})")

# Explanation
- The evaluate_k_range function was run in order to see how many clusters are optimal. 2 centroids/clusters appears to be the optimal k based on the silhouette score, so that is what was used in K_Means clustering algorithm and that is what is displayed on the above PCA graph. 

## 6. Hierarchical Clustering and Linkage Methods

# DENDOGRAM

In [ ]:
max_points_for_dendrogram = 30
sample_df = feature_df.copy()

if len(sample_df) > max_points_for_dendrogram:
    sample_df = sample_df.sample(max_points_for_dendrogram, random_state=RANDOM_STATE)

Z = linkage(sample_df, method="ward")

plt.figure(figsize=(12, 5))
dendrogram(Z, leaf_rotation=90, leaf_font_size=8)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Sample index")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()

# AGGLOMERATIVE CLUSTERING

In [ ]:
results = []
for k in range(2, 9):
    model = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = model.fit_predict(df_scaled)
    score = silhouette_score(df_scaled, labels)
    results.append({"k": k, "silhouette_score": score})

results_df = pd.DataFrame(results)
display(results_df)

plt.figure(figsize=(7, 4))
sns.lineplot(data=results_df, x="k", y="silhouette_score", marker="o")
plt.title("Silhouette Score by Number of Clusters")
plt.xlabel("Number of clusters")
plt.ylabel("Silhouette score")
plt.tight_layout()
plt.show()

In [ ]:
best_k = int(results_df.sort_values("silhouette_score", ascending=False).iloc[0]["k"])
# best_k = 4
print("Selected K:", best_k)

final_model = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
cluster_labels = final_model.fit_predict(df_scaled)

df_hier = df.copy()
df_hier["hier_cluster"] = cluster_labels

print(df_hier["hier_cluster"].value_counts().sort_index())

# Explanation
- k - 3 has the best silhouette so will start with that one. ~~Ended up changing it to 2, because 3 didn't look much better.~~ Changed it back to 3 after generating the linkage table 

In [ ]:
plot_df = pca_df.copy()
plot_df["hier_cluster"] = cluster_labels.astype(str)

plt.figure(figsize=(7, 5))
sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="hier_cluster", s=25)
plt.title("Agglomerative Clustering Result")
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
linkage_results = []

for linkage_name in ["ward", "complete", "average", "single"]:
    for k in range(2, 9):
        model = AgglomerativeClustering(n_clusters=k, linkage=linkage_name)
        labels = model.fit_predict(df_scaled)
        score = silhouette_score(df_scaled, labels)
        linkage_results.append({
            "linkage": linkage_name,
            "k": k,
            "silhouette_score": score
        })

linkage_df = pd.DataFrame(linkage_results)
display(linkage_df.sort_values("silhouette_score", ascending=False).head(10))

plt.figure(figsize=(8, 5))
sns.lineplot(data=linkage_df, x="k", y="silhouette_score", hue="linkage", marker="o")
plt.title("Silhouette Score by Linkage Method")
plt.tight_layout()
plt.show()

# Interpretation
- The linkage table makes 2 through 4 look like reasonable k-values. The ward linkage silhouette score is lower than the others because it is trying to divide up the large dense cluster created by overlapping features. 

# 7. DBSCAN Clustering 

In [ ]:
min_samples = 10

neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors.fit(df_scaled)
distances, indices = neighbors.kneighbors(df_scaled)

k_distances = np.sort(distances[:, min_samples - 1])

plt.figure(figsize=(8, 4))
plt.plot(k_distances)
plt.title("k-Distance Plot for Choosing eps")
plt.xlabel("Points sorted by distance")
plt.ylabel(f"Distance to {min_samples}th nearest neighbor")
plt.tight_layout()
plt.show()

print("Suggested starting eps values to try:")
print(np.round(np.percentile(k_distances, [70, 75, 80, 85, 90, 95]), 3))

In [ ]:
eps = float(np.percentile(k_distances, 85))
print("Starting eps:", round(eps, 3))
print("min_samples:", min_samples)

dbscan = DBSCAN(eps=eps, min_samples=min_samples)
db_labels = dbscan.fit_predict(df_scaled)

df_db = df.copy()
df_db["dbscan_cluster"] = db_labels

display(df_db["dbscan_cluster"].value_counts().sort_index())

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = int((db_labels == -1).sum())

print("Clusters found:", n_clusters)
print("Noise points:", n_noise)

# Compute Silhouette Score

In [ ]:
non_noise_mask = db_labels != -1
non_noise_labels = db_labels[non_noise_mask]

if len(set(non_noise_labels)) >= 2:
    score = silhouette_score(df_scaled[non_noise_mask], non_noise_labels)
    print("Silhouette score on non-noise points:", round(score, 4))
else:
    print("Silhouette score not computed: fewer than two non-noise clusters.")

# Visualize DBSCAN Clusters

In [ ]:
plot_df = pca_df.copy()
plot_df["dbscan_cluster"] = db_labels.astype(str)

plt.figure(figsize=(7, 5))
sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="dbscan_cluster", s=55)
plt.title("DBSCAN Clustering Result")
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Tune eps and min_samples

In [ ]:
eps_values = np.round(np.percentile(k_distances, [70, 75, 80, 85, 90, 95]), 3)
min_samples_values = [4, 5, 8, 10]

sweep_rows = []

for ms in min_samples_values:
    for ep in eps_values:
        model = DBSCAN(eps=float(ep), min_samples=ms)
        labels = model.fit_predict(df_scaled)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = int((labels == -1).sum())
        noise_rate = n_noise / len(labels)

        mask = labels != -1
        if len(set(labels[mask])) >= 2:
            sil = silhouette_score(df_scaled[mask], labels[mask])
        else:
            sil = np.nan

        sweep_rows.append({
            "eps": ep,
            "min_samples": ms,
            "clusters": n_clusters,
            "noise_rate": round(noise_rate, 3),
            "silhouette_non_noise": sil
        })

sweep_df = pd.DataFrame(sweep_rows)
display(sweep_df.sort_values(["silhouette_non_noise", "clusters"], ascending=[False, False]).head(15))

# Select practical DBSCAN Setting

In [ ]:
candidates = sweep_df[
    (sweep_df["clusters"] >= 2) &
    (sweep_df["noise_rate"] < 0.40)
].dropna(subset=["silhouette_non_noise"])

if len(candidates) > 0:
    chosen = candidates.sort_values("silhouette_non_noise", ascending=False).iloc[0]
else:
    chosen = sweep_df.sort_values("clusters", ascending=False).iloc[0]

best_eps = float(chosen["eps"])
best_min_samples = int(chosen["min_samples"])

print("Chosen eps:", best_eps)
print("Chosen min_samples:", best_min_samples)

final_dbscan = DBSCAN(eps=best_eps, min_samples=best_min_samples)
final_labels = final_dbscan.fit_predict(df_scaled)

df_db["dbscan_cluster_final"] = final_labels
display(df_db["dbscan_cluster_final"].value_counts().sort_index())

# Inspect Noise

In [ ]:
profile_df = feature_df.copy()
profile_df["dbscan_cluster_final"] = final_labels

profile = profile_df.groupby("dbscan_cluster_final").mean().round(2)
display(profile)

plt.figure(figsize=(10, 5))
sns.heatmap(profile, cmap="Blues", annot=False)
plt.title("DBSCAN Cluster and Noise Feature Profiles")
plt.xlabel("Feature")
plt.ylabel("Cluster (-1 means noise)")
plt.tight_layout()
plt.show()

# Explanation

- The dbscan did not seem to work well for this dataset. There are too many features and the features have too similar of flags. 

In [ ]:
def cluster_summary(name, params, labels, notes, has_noise=False):
    labels = np.asarray(labels)
    noise = int((labels == -1).sum())
    mask = labels != -1
    n_clusters = len(set(labels[mask]))
    if n_clusters >= 2:
        sil = round(silhouette_score(df_scaled[mask], labels[mask]), 3)
    else:
        sil = np.nan
    return {
        "method": name,
        "Parameters Used": params,
        "number of clusters": n_clusters,
        "noise_points": noise if has_noise else "N/A",
        "silhouette": sil,
        "Interpretation": notes,
    }

summary_rows = [
    cluster_summary(
        "K-Means", "k=2", df["kmeans_cluster"].values,
        f"K={CHOSEN_K} chosen by silhouette"),
    cluster_summary(
        "Hierarchical (Ward)", "k=3",df_hier["hier_cluster"].values,
        f"K={best_k} one dominating dense cluster"),
    cluster_summary(
        "DBSCAN","eps=0.3, min_samples=10", final_labels,
        f"eps={best_eps}, min_samples={best_min_samples}. Lot of micro clusters and noise",
        has_noise=True),
]

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

# Interpretation

K-means clustering provided the best results for this dataset. Using a k-value of 2 the results were the clearest. The hierarchical clustering had a low silhouette score and made a very dense cluster. DBSCAN was unable to meaningfully separate all of the data, because it had such high dimensionality. 